# Advanced GNN Architectures

## Beyond Standard GNNs: Next-Level Concepts

This notebook explores sophisticated GNN architectures for specialized problems:
- GraphSAGE (inductive learning)
- Graph Isomorphism Networks (GIN)  
- Heterogeneous GNNs
- Temporal/Dynamic Graphs
- Graph Pooling and Coarsening
- Scalability techniques

### Prerequisites
- Complete Module 1 (Fundamentals)
- Understanding of message passing framework
- Familiarity with attention mechanisms

---

# GraphSAGE: Inductive Learning with Sampling

## Motivation

**Limitations of standard GCN/GAT:**
- Transductive: Can't generate embeddings for new nodes
- Requires full graph in memory
- Doesn't scale to very large graphs

**GraphSAGE Solution:**
- **Inductive**: Learn on training nodes, generalize to unseen nodes
- **Mini-batch sampling**: Memory-efficient training
- **Flexible aggregation**: Multiple aggregator options

## Mathematical Formulation

For each node $v$ at layer $\ell$:

### Step 1: Sample neighbors
$$\mathcal{N}_\ell^s(v) \subseteq \mathcal{N}(v), \quad |\mathcal{N}_\ell^s(v)| = S_\ell$$

Sample at most $S_\ell$ neighbors (uniform random sampling)

### Step 2: Aggregate neighbor features  
$$m_v = \text{AGGREGATE}(\{h_u^{(\ell-1)} : u \in \mathcal{N}_\ell^s(v)\})$$

Options:
- **Mean**: $m_v = \frac{1}{|N_\ell^s(v)|} \sum_{u \in N_\ell^s(v)} h_u$
- **LSTM**: Process neighbor sequence with LSTM
- **Pooling**: $m_v = \text{MAX}(MLP(h_u))$ for all neighbors

### Step 3: Update with own features
$$h_v^{(\ell)} = \text{ReLU}(W[\text{concat}(h_v^{(\ell-1)}, m_v)])$$

## Computational Complexity

**Standard GCN (single forward pass):**
- Forward: $O(|E| \cdot d_{hidden})$
- Memory: $O(|V| \cdot d_{hidden})$

**GraphSAGE with sampling:**
- Per-layer sampling cost: $O(|batch| \cdot S_\ell \cdot d_{hidden})$
- For $K$ layers: $O(|batch| \cdot \prod_\ell S_\ell \cdot d_{hidden})$
- Memory per layer: $O(|batch| \cdot S_\ell \cdot d_{hidden})$

**Key advantage**: Decouples computation from graph size!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt

# ============================================================================
# GraphSAGE Implementation
# ============================================================================

class MeanAggregator(nn.Module):
    """Mean aggregator for GraphSAGE."""
    
    def forward(self, neighbor_features: torch.Tensor) -> torch.Tensor:
        """
        Aggregate via mean.
        
        Args:
            neighbor_features: (batch_size * num_neighbors, feature_dim)
            
        Returns:
            aggregated: (batch_size, feature_dim)
        """
        # Reshape and average
        # In practice would need to handle variable numbers of neighbors
        return neighbor_features.mean(dim=0, keepdim=True)


class GraphSAGELayer(nn.Module):
    """
    GraphSAGE layer with neighbor sampling and aggregation.
    
    Key ideas:
    1. Sample neighbors (don't use all)
    2. Aggregate features from samples  
    3. Combine with own features
    """
    
    def __init__(self, in_channels: int, out_channels: int,
                 aggregator: str = 'mean', sample_size: int = 10):
        """
        Initialize GraphSAGE layer.
        
        Args:
            in_channels: Input feature dimension
            out_channels: Output feature dimension
            aggregator: Aggregation method ('mean', 'lstm', 'pool')
            sample_size: Number of neighbors to sample
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.sample_size = sample_size
        self.aggregator = aggregator
        
        # Neighbor aggregation transformation
        if aggregator == 'mean':
            self.agg_func = nn.Linear(in_channels, out_channels)
        elif aggregator == 'lstm':
            self.agg_func = nn.LSTM(in_channels, out_channels, batch_first=True)
        elif aggregator == 'pool':
            self.agg_func = nn.Sequential(
                nn.Linear(in_channels, out_channels),
                nn.ReLU(),
                nn.Linear(out_channels, out_channels)
            )
        
        # Self-feature transformation
        self.self_transform = nn.Linear(in_channels, out_channels)
        
        # Combination layer
        self.combine = nn.Linear(2 * out_channels, out_channels)
    
    def forward(self, node_features: torch.Tensor, 
               neighbor_features_list: List[torch.Tensor]) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            node_features: (batch_size, in_channels)
            neighbor_features_list: List of (batch_size, in_channels) tensors
            
        Returns:
            output: (batch_size, out_channels)
        """
        batch_size = node_features.size(0)
        
        # Transform self features
        self_features = self.self_transform(node_features)  # (batch, out_channels)
        
        # Aggregate neighbor features
        if len(neighbor_features_list) == 0:
            neighbor_agg = torch.zeros(batch_size, self.out_channels,
                                      device=node_features.device)
        else:
            # Stack neighbor features
            neighbor_stack = torch.stack(neighbor_features_list, dim=1)  
            # (batch, num_neighbors, in_channels)
            
            if self.aggregator == 'mean':
                neighbor_agg = torch.mean(neighbor_stack, dim=1)  
                # (batch, in_channels)
                neighbor_agg = self.agg_func(neighbor_agg)
                
            elif self.aggregator == 'lstm':
                # Use LSTM to process neighbor sequence
                _, (h_n, _) = self.agg_func(neighbor_stack)
                neighbor_agg = h_n.squeeze(0)  # (batch, out_channels)
                
            elif self.aggregator == 'pool':
                # Max-pool over neighbors
                neighbor_agg = torch.max(
                    torch.stack([self.agg_func(n) for n in neighbor_stack], dim=1),
                    dim=1
                )[0]  # (batch, out_channels)
        
        # Combine self and neighbor representations
        combined = torch.cat([self_features, neighbor_agg], dim=1)  
        # (batch, 2*out_channels)
        
        output = self.combine(combined)  # (batch, out_channels)
        output = F.relu(output)
        
        return output


class GraphSAGEModel(nn.Module):
    """
    Full GraphSAGE model for inductive learning.
    """
    
    def __init__(self, in_channels: int, hidden_channels: int,
                 num_classes: int, num_layers: int = 2,
                 aggregator: str = 'mean', sample_sizes: List[int] = None):
        """
        Initialize GraphSAGE model.
        
        Args:
            in_channels: Input feature dimension
            hidden_channels: Hidden layer dimension
            num_classes: Output classes
            num_layers: Number of layers
            aggregator: Aggregation method
            sample_sizes: Sampling sizes per layer
        """
        super().__init__()
        self.num_layers = num_layers
        
        if sample_sizes is None:
            sample_sizes = [10] * num_layers
        
        self.sample_sizes = sample_sizes
        
        # Layers
        self.layers = nn.ModuleList()
        
        # First layer
        self.layers.append(GraphSAGELayer(in_channels, hidden_channels,
                                         aggregator, sample_sizes[0]))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.layers.append(GraphSAGELayer(hidden_channels, hidden_channels,
                                             aggregator, sample_sizes[_]))
        
        # Output layer
        self.layers.append(GraphSAGELayer(hidden_channels, num_classes,
                                         aggregator, sample_sizes[-1]))
    
    def forward(self, node_features: torch.Tensor,
               sampled_neighbors: List[List[int]]) -> torch.Tensor:
        """
        Forward with sampled neighbors.
        
        Args:
            node_features: (V, in_channels)
            sampled_neighbors: Multi-hop sampled neighbors
            
        Returns:
            logits: (batch_size, num_classes)
        """
        # This is a simplified version
        # In practice, would implement proper sampling and batching
        x = node_features
        for layer in self.layers:
            # Simulate aggregation (simplified)
            x = layer(x, [])
        return x

print("✓ GraphSAGE implementation loaded!")
print("""
Key Features:
1. Neighbor sampling reduces computation
2. Multiple aggregators (mean, LSTM, pool)
3. Inductive learning capability
4. Scalable to large graphs
""")